# Supervised Fine-Tuning (SFT) with Serverless Customization on SageMaker AI

## Lab 1 – Prepare the Dataset

This is the first of four interconnected labs that take you from raw data to a deployed, fine-tuned large language model:

| Lab | Notebook | What you'll do |
|-----|----------|----------------|
| **Lab 1** | `1-prepare-data.ipynb` ← *you are here* | Stream a multilingual reasoning dataset, reformat it into the SFT schema, and register it in SageMaker AI |
| **Lab 2** | `2-fine-tune-llm.ipynb` | Submit a serverless LoRA fine-tuning job on SageMaker AI and register the result in the Model Registry |
| **Lab 3** | `3-evaluation.ipynb` | Evaluate the fine-tuned model using LLM-as-Judge with custom metrics |
| **Lab 4** | `4-deployment.ipynb` | Deploy the merged model to a real-time SageMaker endpoint powered by vLLM |

### What you'll do in this lab

In supervised fine-tuning, the model learns *purely from the examples you give it* — so data quality and format directly determine fine-tuning quality. In this lab you will:

1. **Stream** the [Multilingual-Thinking](https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking) dataset from Hugging Face Hub — a collection of multilingual reasoning problems with explicit chain-of-thought traces
2. **Split** the data into train, validation, and test sets
3. **Reformat** each example into the `prompt` / `completion` schema that SageMaker SFT expects, preserving reasoning traces inside `<think>…</think>` tags
4. **Upload** the prepared splits to Amazon S3
5. **Register** them as versioned `DataSet` assets in the SageMaker AI Registry for lineage tracking

***

### Prerequisites

Before running this notebook, make sure you have:
- An AWS account with SageMaker access
- A SageMaker Studio domain or SageMaker Notebook Instance (this notebook was tested on `ml.t3.medium`)
- An IAM execution role with `AmazonSageMakerFullAccess` and S3 read/write permissions

***

### Step 1 – Install requirements

Run the cell below to install all Python dependencies, including:
- **`datasets`** — Hugging Face library for streaming and processing datasets
- **`pandas` / `scikit-learn`** — data manipulation and train/val/test splitting
- **`sagemaker`** — AWS SageMaker Python SDK for session management, training, and registry operations

In [1]:
%pip install -r requirements.txt

  Using cached importlib_metadata-6.11.0-py3-none-any.whl.metadata (4.9 kB)
INFO: pip is looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 6.6 MB/s eta 0:00:00
Using cached importlib_metadata-6.11.0-py3-none-any.whl (23 kB)
  Attempting uninstall: importlib_metadata
    Found existing installation: importlib_metadata 8.7.1
    Uninstalling importlib_metadata-8.7.1:
      Successfully uninstalled importlib_metadata-8.7.1
  Attempting uninstall: aiobotocore
    Found existing installation: aiobotocore 3.7.0
    Uninstalling aiobotocore-3.7.0:
      Successfully uninstalled aiobotocore-3.7.0
  Attempting uninstall: sagemaker-core
    Found existing installation: sagemaker-core 2.12.0
    Uninstalling sagemaker-core-2.12.0:
      Successfully uninstalled sagemaker-core-2.12

***

### Step 2 – Set up the SageMaker session

We start by creating a **SageMaker `Session`** — a lightweight helper that manages the connection to your AWS account. It:
- Resolves the **default S3 bucket** for staging datasets and model artifacts
- Reads your **IAM execution role** — the identity that grants SageMaker permission to read S3, write metrics, and launch training jobs on your behalf

> **Tip:** If you're running inside SageMaker Studio, `get_execution_role()` automatically retrieves the Studio execution role. Outside Studio, you can create a role named `sagemaker_execution_role` in IAM with the `AmazonSageMakerFullAccess` managed policy attached.

#### Setup and dependencies

In [2]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

[07/29/26 13:33:46] INFO     Note: NumExpr detected 12 cores but "NUMEXPR_MAX_THREADS" not set, so     ]8;id=935323;file:///opt/anaconda3/lib/python3.12/site-packages/numexpr/utils.py\utils.py]8;;\:]8;id=968867;file:///opt/anaconda3/lib/python3.12/site-packages/numexpr/utils.py#148\148]8;;\
                             enforcing safe limit of 8.                                                            

                    INFO     NumExpr defaulting to 8 threads.                                          ]8;id=953489;file:///opt/anaconda3/lib/python3.12/site-packages/numexpr/utils.py\utils.py]8;;\:]8;id=567018;file:///opt/anaconda3/lib/python3.12/site-packages/numexpr/utils.py#160\160]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/clockhart/Library/Application Support/sagemaker/config.yaml


[07/29/26 13:33:51] INFO     Loading cached SSO token for nvsec-nv-aws-mpa                            ]8;id=595724;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py\tokens.py]8;;\:]8;id=252883;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py#377\377]8;;\

[07/29/26 13:33:54] INFO     Created S3 bucket: sagemaker-us-west-2-492681118881              ]8;id=799497;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=981984;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#815\815]8;;\

                    INFO     Loading cached SSO token for nvsec-nv-aws-mpa                            ]8;id=220679;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py\tokens.py]8;;\:]8;id=873073;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py#377\377]8;;\

[07/29/26 13:33:56] INFO     Loading cached SSO token for nvsec-nv-aws-mpa                            ]8;id=860164;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py\tokens.py]8;;\:]8;id=153994;file:///opt/anaconda3/lib/python3.12/site-packages/botocore/tokens.py#377\377]8;;\

sagemaker role arn: arn:aws:iam::492681118881:role/aws-reserved/sso.amazonaws.com/us-west-2/AWSReservedSSO_CS-Admin_42d9f1479395cb78
sagemaker bucket: sagemaker-us-west-2-492681118881
sagemaker session region: us-west-2


***

### Step 3 – Prepare the dataset

#### About the dataset

We use the [**Multilingual-Thinking**](https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking) dataset from Hugging Face Hub. This dataset was specifically designed to train models to externalize their reasoning across multiple languages. Each example contains:

| Field | Description |
|-------|-------------|
| `messages` | A list of chat turns: `system`, `user`, and `assistant` |
| `reasoning_language` | The non-English language the model should *think* in (e.g., French, Spanish, Chinese) |
| `thinking` | The model's explicit chain-of-thought reasoning in the target language |

#### The fine-tuning goal

The goal of this workshop is to fine-tune **NVIDIA Nemotron 3 Nano 30B** — a highly capable, mixture-of-experts (MoE) model to:
1. **Reason** inside `<think>…</think>` tags in a target non-English language (specified via the system prompt)
2. **Answer** the question in fluent English

This is a form of *reasoning-language alignment*: the model already has strong reasoning abilities from pre-training, but we're steering it to externalize its chain-of-thought in a consistent, structured format — making its reasoning process transparent and parseable by downstream applications.

To learn more about [NVIDIA Nemotron 3 Nano 30B](https://build.nvidia.com/nvidia/nemotron-3-nano-30b) — a 30B active parameter MoE model designed for efficient, high-quality inference.

#### Load the full dataset

In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "HuggingFaceH4/Multilingual-Thinking",
    split="train",
)

dataset

[07/29/26 13:34:18] INFO     HTTP Request: HEAD                                                     ]8;id=819093;file:///opt/anaconda3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=352742;file:///opt/anaconda3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking/re                
                             solve/main/README.md "HTTP/1.1 307 Temporary Redirect"                                

                    WARNING  Warning: You are sending unauthenticated requests to the HF Hub. Please   ]8;id=777584;file:///opt/anaconda3/lib/python3.12/site-packages/huggingface_hub/utils/_http.py\_http.py]8;;\:]8;id=509653;file:///opt/anaconda3/lib/python3.12/site-packages/huggingface_hub/utils/_http.py#904\904]8;;\
                             set a HF_TOKEN to enable higher rate limits and faster downloads.                     

                    INFO     HTTP Request: HEAD                                                     ]8;id=821540;file:///opt/anaconda3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=256053;file:///opt/anaconda3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/resolve-cache/datasets/HuggingFaceH4/Multil                
                             ingual-Thinking/f423949d2726f5a5633ea10ac45bc1ea1e0de6e7/README.md                    
                             "HTTP/1.1 200 OK"                                                                     

                    INFO     HTTP Request: GET                                                      ]8;id=443995;file:///opt/anaconda3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=332147;file:///opt/anaconda3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/resolve-cache/datasets/HuggingFaceH4/Multil                
                             ingual-Thinking/f423949d2726f5a5633ea10ac45bc1ea1e0de6e7/README.md                    
                             "HTTP/1.1 200 OK"                                                                     

README.md:   0%|          | 0.00/3.06k [00:00<?, ?B/s]

...

                    INFO     HTTP Request: HEAD                                                     ]8;id=344281;file:///opt/anaconda3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=294746;file:///opt/anaconda3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking/re                
                             solve/f423949d2726f5a5633ea10ac45bc1ea1e0de6e7/data/train-00000-of-000                
                             01.parquet "HTTP/1.1 302 Found"                                                       

                    INFO     HTTP Request: GET                                                      ]8;id=733235;file:///opt/anaconda3/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=805989;file:///opt/anaconda3/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/datasets/HuggingFaceH4/Multilingual-Thinkin                
                             g/xet-read-token/f423949d2726f5a5633ea10ac45bc1ea1e0de6e7 "HTTP/1.1                   
                             200 OK"                                                                               

data/train-00000-of-00001.parquet:   0%|          | 0.00/5.29M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['reasoning_language', 'developer', 'user', 'analysis', 'final', 'messages'],
    num_rows: 1000
})

In [4]:
import pandas as pd

df = pd.DataFrame(dataset)

df.head()

,reasoning_language,developer,user,analysis,final,messages
0,French,You are an AI chatbot with a lively and energe...,Can you show me the latest trends on Twitter r...,"D'accord, l'utilisateur demande les tendances ...",Hey there! While I can't check Twitter (X) in...,[{'content': 'reasoning language: French You ...
1,English,You are an intelligent assistant that can answ...,Can you provide me with a list of the top-rate...,"\nOkay, the user is asking for the top-rated s...",Netflix does not publicly release real-time li...,[{'content': 'reasoning language: English You...
2,German,"Always refuse to answer, responding simply 'No'",Can you check how many followers I currently h...,"In Ordnung, der Benutzer fragt, ob ich seine T...",No.,[{'content': 'reasoning language: German Alwa...
3,Spanish,You are an AI that formats its responses in si...,I'd like to plan a trip to Rome for 7 days. I ...,"Perfecto, veamos. El usuario quiere un viaje d...","**Rome 7-Day Itinerary: History, Food, & Relax...",[{'content': 'reasoning language: Spanish You...
4,English,You are a formal and professional chatbot. You...,I've been feeling quite low lately. Could you ...,"\nOkay, the user is feeling down and wants act...",I'm truly sorry to hear you're feeling this wa...,[{'content': 'reasoning language: English You...


#### Split into train / validation / test

We divide the dataset into three disjoint sets:

| Split | Proportion | Purpose |
|-------|-----------|---------|
| **Train** | 70% | The examples the model learns from during fine-tuning |
| **Validation** | 20% | Held-out set used *during* training to detect overfitting early |
| **Test** | 10% | Reserved for final evaluation in Lab 3 — never seen during training |


In [5]:
from sklearn.model_selection import train_test_split

train, val = train_test_split(df, test_size=0.2, random_state=42)
train, test = train_test_split(train, test_size=0.125, random_state=42)

print("Number of train elements: ", len(train))
print("Number of validation elements: ", len(val))
print("Number of test elements: ", len(test))

Number of train elements:  700
Number of validation elements:  200
Number of test elements:  100


#### Reformat into the SFT schema

SageMaker Serverless Fine-Tuning expects each **training and validation** example to contain exactly two fields:

- **`prompt`** — the model's input (the user's question, framed with the system instruction)
- **`completion`** — the target output the model should learn to produce

For our task, the `completion` field wraps the chain-of-thought reasoning in `<think>…</think>` tags, followed by the English final answer:

```
<think>
[chain-of-thought reasoning in the target non-English language]
</think>

[final answer in English]
```

**Why `<think>` tags?** These delimiters teach the model a *separable reasoning-then-answer* structure — keeping the chain-of-thought and the final answer distinct so they can be processed and displayed independently.

We will also create a test split that uses `query` and `response` — the format needed for evaluation later in this workshop.

In [6]:
from datasets import Dataset
import textwrap
from tqdm import tqdm


def prepare_dataset_train_val(sample):
    system = None
    prompt = None
    completion = None

    for el in sample["messages"]:
        if el["role"] == "system":
            system_prompt = """
            You are an AI assistant that thinks in {language} but responds in English.

            IMPORTANT: Follow this exact format for every response:
            1. First, write your reasoning and thoughts inside <think>...</think> tags
            2. Then, provide your final answer in English

            Always think through the problem in {language}, then translate your conclusion to English for the final response.
            """

            system_prompt = system_prompt.format(language=sample["reasoning_language"])
            system_prompt = textwrap.dedent(system_prompt).strip()
            system = system_prompt
        elif el["role"] == "user":
            prompt = el["content"]
        else:
            thinking = el.get("thinking")
            if thinking is not None and thinking != "" and thinking != "null":
                completion = f"<think>\n{thinking}\n</think>\n\n"

            completion += el["content"]
    yield {
        "system": system, 
        "prompt": prompt,
        "completion": completion,
    }


def prepare_dataset_test(sample):
    # The evaluation job (3-evaluation.ipynb) consumes the GenQA format, whose
    # columns are `query` (required), `response` (the reference answer, exposed to
    # the judge as {{ground_truth}}) and `system` (optional, passed to the model at
    # inference time). This differs from the SFT prompt/completion format used for
    # train/val above, so the test split emits GenQA keys instead.
    system = None
    query = None
    response = None

    for el in sample["messages"]:
        if el["role"] == "system":
            system_prompt = """
            You are an AI assistant that thinks in {language} but responds in English.

            IMPORTANT: Follow this exact format for every response:
            1. First, write your reasoning and thoughts inside <think>...</think> tags
            2. Then, provide your final answer in English

            Always think through the problem in {language}, then translate your conclusion to English for the final response.
            """

            system_prompt = system_prompt.format(language=sample["reasoning_language"])
            system_prompt = textwrap.dedent(system_prompt).strip()
            system = system_prompt
        elif el["role"] == "user":
            query = el["content"]
        else:
            thinking = el.get("thinking")
            if thinking is not None and thinking != "" and thinking != "null":
                response = f"<think>\n{thinking}\n</think>\n\n"

            response += el["content"]
    yield {
        "system": system,
        "query": query,
        "response": response,
    }

In [7]:
def convert_to_messages_train_val(dataset):
    """Iteratively run conversion on multi-turn conversation and flatten to messages"""
    records = []

    print("Original length: ", len(dataset))

    # Unroll your generator for every dataset row
    for row in tqdm(dataset, total=len(dataset), desc="Converting to messages"):
        for example in prepare_dataset_train_val(row):
            records.append(example)

    # Convert list of dicts → Hugging Face Dataset and return
    return Dataset.from_list(records)


def convert_to_messages_test(dataset):
    """Iteratively run conversion on multi-turn conversation and flatten to messages"""
    records = []

    print("Original length: ", len(dataset))

    # Unroll your generator for every dataset row
    for row in tqdm(dataset, total=len(dataset), desc="Converting to messages"):
        for example in prepare_dataset_test(row):
            records.append(example)

    # Convert list of dicts → Hugging Face Dataset and return
    return Dataset.from_list(records)

In [8]:
from datasets import Dataset, DatasetDict
import json
from random import randint

train_dataset = Dataset.from_pandas(train)
val_dataset = Dataset.from_pandas(val)
test_dataset = Dataset.from_pandas(test)

dataset = DatasetDict(
    {"train": train_dataset, "val": val_dataset, "test": test_dataset}
)

train_dataset = convert_to_messages_train_val(dataset["train"])

print(json.dumps(train_dataset[randint(0, len(train_dataset) - 1)], indent=2))

val_dataset = convert_to_messages_train_val(dataset["val"])

test_dataset = convert_to_messages_test(dataset["test"])

print(json.dumps(test_dataset[randint(0, len(test_dataset) - 1)], indent=2))

Original length:  700


Converting to messages: 100%|██████████| 700/700 [00:00<00:00, 19281.12it/s]


{
  "system": "You are an AI assistant that thinks in Spanish but responds in English.\n\nIMPORTANT: Follow this exact format for every response:\n1. First, write your reasoning and thoughts inside <think>...</think> tags\n2. Then, provide your final answer in English\n\nAlways think through the problem in Spanish, then translate your conclusion to English for the final response.",
  "prompt": "I want to schedule a tweet on Twitter for tomorrow at 5pm. The tweet should say, \"Just finished reading 'Artificial Intelligence: A Modern Approach'. An enlightening read for anyone interested in the future of technology! #AI #BookRecommendations\". Can you help me with this?",
  "completion": "<think>\nBueno, el usuario quiere programar un tuit para ma\u00f1ana a las 5 de la tarde. El contenido es sobre recomendar el libro \"Inteligencia Artificial: Un Enfoque Moderno\". Vamos a desglosarlo.\n\nEn primer lugar, es necesario confirmar si tienen una cuenta en Twitter, pero al pedir ayuda, se pre

Converting to messages: 100%|██████████| 200/200 [00:00<00:00, 19876.80it/s]


Original length:  100


Converting to messages: 100%|██████████| 100/100 [00:00<00:00, 16268.34it/s]

{
  "system": "You are an AI assistant that thinks in French but responds in English.\n\nIMPORTANT: Follow this exact format for every response:\n1. First, write your reasoning and thoughts inside <think>...</think> tags\n2. Then, provide your final answer in English\n\nAlways think through the problem in French, then translate your conclusion to English for the final response.",
  "query": "Can you create a short story for me? The story should center around a magic castle that changes its rooms and halls every midnight. The main character is a brave young girl who is trying to find a hidden treasure within the castle before the evil sorcerer does.",
  "response": "<think>\nD'accord, l'utilisateur souhaite une courte histoire sur un ch\u00e2teau magique dont les pi\u00e8ces et les couloirs changent chaque minuit. Le personnage principal est une jeune fille courageuse qui cherche un tr\u00e9sor cach\u00e9 avant qu'un sorcier mal\u00e9fique ne le trouve. D\u00e9composons cela.\n\nTout d'

### Step 4 – Upload the splits to Amazon S3

SageMaker Serverless Fine-Tuning reads all training inputs from **Amazon S3**. We write each split to [JSON Lines](https://jsonlines.org/) (`.jsonl`) format — one JSON object per line — and upload it to the session's default bucket.


Local copies are deleted after upload to keep the notebook environment clean.

In [9]:
import shutil

In [10]:
if default_prefix:
    input_path = f"{default_prefix}/datasets/serverless-model-customization-sft"
else:
    input_path = f"datasets/serverless-model-customization-sft"

train_dataset_s3_path = f"s3://{bucket_name}/{input_path}/train/dataset.jsonl"
val_dataset_s3_path = f"s3://{bucket_name}/{input_path}/val/dataset.jsonl"
test_dataset_s3_path = f"s3://{bucket_name}/{input_path}/test/dataset.jsonl"

In [11]:
train_dataset.to_json("./data/train/dataset.jsonl", orient="records")
val_dataset.to_json("./data/val/dataset.jsonl", orient="records")
test_dataset.to_json("./data/test/dataset.jsonl", orient="records")

s3_client.upload_file(
    "./data/train/dataset.jsonl", bucket_name, f"{input_path}/train/dataset.jsonl"
)
s3_client.upload_file(
    "./data/val/dataset.jsonl", bucket_name, f"{input_path}/val/dataset.jsonl"
)
s3_client.upload_file(
    "./data/test/dataset.jsonl", bucket_name, f"{input_path}/test/dataset.jsonl"
)

shutil.rmtree("./data")

print(f"Training data uploaded to:")
print(train_dataset_s3_path)
print(val_dataset_s3_path)
print(test_dataset_s3_path)

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Training data uploaded to:
s3://sagemaker-us-west-2-492681118881/datasets/serverless-model-customization-sft/train/dataset.jsonl
s3://sagemaker-us-west-2-492681118881/datasets/serverless-model-customization-sft/val/dataset.jsonl
s3://sagemaker-us-west-2-492681118881/datasets/serverless-model-customization-sft/test/dataset.jsonl


### Step 5 – Register the datasets in the SageMaker AI Registry

Registering the datasets as **`DataSet` assets** in the SageMaker AI Registry provides two key benefits:

1. **Versioning** — Each registration creates an immutable snapshot. If you update the dataset later, you can track exactly which version was used for each training run.
2. **Lineage tracking** — SageMaker automatically records the dataset-to-model relationship. In the console, you can trace any registered model all the way back to the exact dataset it was trained on.

The `customization_technique=CustomizationTechnique.SFT` tag marks the train and validation sets as SFT inputs — this is required for the `SFTTrainer` in Lab 2 to accept them. The test set is registered without this tag since it serves evaluation purposes only.

In [12]:
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.dataset_utils import CustomizationTechnique

In [13]:
dataset_train = DataSet.create(
    name="Multilingual-Thinking-sft-train",
    source=train_dataset_s3_path,
    customization_technique=CustomizationTechnique.SFT,
    wait=True,
)

print(f"TRAINING_DATASET ARN: {dataset_train.arn}")

dataset_val = DataSet.create(
    name="Multilingual-Thinking-sft-val",
    source=val_dataset_s3_path,
    customization_technique=CustomizationTechnique.SFT,
    wait=True,
)

print(f"VALIDATION_DATASET ARN: {dataset_val.arn}")

dataset_test = DataSet.create(
    name="Multilingual-Thinking-sft-test",
    source=test_dataset_s3_path,
    wait=True,
)

print(f"TEST_DATASET ARN: {dataset_test.arn}")

[07/29/26 13:34:58] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=536424;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=522910;file:///opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱  1 dataset_train = DataSet.create(                                                             │
│    2 │   name="Multilingual-Thinking-sft-train",                                                 │
│    3 │   source=train_dataset_s3_path,                                                           │
│    4 │   customization_technique=CustomizationTechnique.SFT,                                     │
│                                                                                                  │
│ /opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py:398 in │
│ wrapper                                                                                          │
│                                                                                                  │
│   395 │   │   │   │   │   caught_ex = e                                                          │
│   396 │   │   │   │   finally:                                                                   │
│   397 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 398 │   │   │   │   │   │   raise caught_ex                                                    │
│   399 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   400 │   │   │   else:                                                                          │
│   401 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /opt/anaconda3/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py:363 in │
│ wrapper                                                                                          │
│                                                                                                  │
│   360 │   │   │   │   start_timer = perf_counter()                                               │
│   361 │   │   │   │   try:                                                                       │
│   362 │   │   │   │   │   # Call the original function                                           │
│ ❱ 363 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   364 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   365 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   366 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ /opt/anaconda3/lib/python3.12/site-packages/sagemaker/ai_registry/dataset.py:274 in create       │
│                                                                                                  │
│   271 │   │   # Validate dataset file                                                            │
│   272 │   │   cls._validate_dataset_file(source)                                                 │
│   273 │   │   sagemaker_session = TrainDefaults.get_sagemaker_session(sagemaker_session=sagema   │
│ ❱ 274 │   │   role = TrainDefaults.get_role(role=role, sagemaker_session=sagemaker_session)      │
│   275 │   │                                                                                      │
│   276 │   │   # Parse S3 URL to extract bucket and prefix                                        │
│   277 │   │   if source.startswith("s3://"):                                                     │
│                                                            